In [25]:
import polars as pl
import pandas as pd
import numpy as mp
import altair as alt
import pyarrow
from func.UsefulFunctions import *

This notebook documents the cleaning and inspection process for the artist data on wikidata

In [26]:
artistDF = pl.read_parquet("./Data/pipelinesArtistsWikidata.parquet")
print(f"There are {len(artistDF["artist"].unique())} unique artist entities after the removal of the disallowed occupations and those without genders labelled.")
display(artistDF.head(3))

There are 444848 unique artist entities after the removal of the disallowed occupations and those without genders labelled.


artist,occupation,gender,occupationLabel,genderLabel,countryCitizenship,influencedBY,influencedBYLabel,statementCount,genre,article,musicBrainzID,recordLabel,recordLabelLabel,countryCitizenshipLabel,genreLabel,artistLabel
str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str
"""Q7161""","""Q36834""","""Q6581097""","""composer""","""male""","""Q29""",null,null,null,"""Q1413570""","""https://en.wikipedia.org/wiki/…","""533761c0-61cd-4d0e-b264-c71d5d…",null,null,"""Spain""","""sardana""","""José María Ventura Casas"""
"""Q7241""","""Q36834""","""Q6581097""","""composer""","""male""","""Q129286""",null,null,538.0,null,"""https://en.wikipedia.org/wiki/…","""13274b60-181b-431a-b049-b48560…",null,null,"""British Raj""",null,"""Rabindranath Tagore"""
"""Q7241""","""Q36834""","""Q6581097""","""composer""","""male""","""Q129286""",null,null,538.0,null,"""https://en.wikipedia.org/wiki/…","""13274b60-181b-431a-b049-b48560…",null,null,"""British Raj""",null,"""Rabindranath Tagore"""


In [27]:
#How many individuals do not have a name in english
noNames = artistDF.filter(
    pl.col("artist").is_in(pl.col("artistLabel").implode())
).select(pl.col("artist")).unique()
display(noNames)
print(f"There are {len(noNames)} artists with no default labels. These were cleaned out.")
noNames.write_csv("./missingValuesStore/artistsEntitiesWithoutLabels.csv")

artist
str
"""Q123348994"""
"""Q19205555"""
"""Q60839067"""
"""Q139426473"""
"""Q12121948"""
…
"""Q139427824"""
"""Q86002396"""
"""Q12213909"""


There are 14134 artists with no default labels. These were cleaned out.


In [28]:
cleanedArtistDF = artistDF.filter(
    ~pl.col("artist").is_in(pl.col("artistLabel").implode())
)
#Add a column for presence of wikiPage
cleanedArtistDF = cleanedArtistDF.with_columns(
    isThereAWiki = pl.col("article").is_not_null()
)
#Finally de-duplicate the data.
cleanedArtistDF = cleanedArtistDF.unique()
display(cleanedArtistDF.tail(3))

artist,occupation,gender,occupationLabel,genderLabel,countryCitizenship,influencedBY,influencedBYLabel,statementCount,genre,article,musicBrainzID,recordLabel,recordLabelLabel,countryCitizenshipLabel,genreLabel,artistLabel,isThereAWiki
str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str,bool
"""Q99676121""","""Q639669""","""Q6581072""","""musician""","""female""",null,null,null,null,null,null,"""7eaf12fb-1064-4558-bae4-c32e83…",null,null,null,null,"""Barbara Greene""",false
"""Q6065692""","""Q12377274""","""Q6581097""","""trumpeter""","""male""","""Q34""",null,null,null,null,null,"""45103d35-94a6-4e5f-af0f-efbe7d…",null,null,"""Sweden""",null,"""Gösta Redlig""",false
"""Q1337866""","""Q15981151""","""Q6581097""","""jazz musician""","""male""","""Q30""",null,null,null,"""Q8341""","""https://en.wikipedia.org/wiki/…","""865567ff-781c-4334-b78d-a235a1…",null,null,"""United States""","""jazz""","""Sonny White""",true


That is the end of the cleaning stage. Now time to analyse the data. Null values were left intentionally as we are looking for the gaps

In [32]:
#Discover the gender gap in the set.
artistGender = cleanedArtistDF.select(
    pl.col("artist"),
    pl.col("gender"),
    pl.col("genderLabel"),
)
artistGender = artistGender.unique()
artistGenderGrouped = artistGender.group_by(["gender", "genderLabel"]).agg(pl.col("artist").count())
artistGenderGroupedFRAC = artistGender["genderLabel"].value_counts(sort=True,normalize=True,name="fraction")
display(artistGenderGrouped)
display(artistGenderGroupedFRAC)

gender,genderLabel,artist
str,str,u32
"""Q301702""","""two-spirit""",14
"""Q7677449""","""takatāpui""",3
"""Q48270""","""non-binary""",472
"""Q18116794""","""genderfluid""",56
"""Q1097630""","""intersex""",8
…,…,…
"""Q93954933""","""demiboy""",1
"""Q17148251""","""travesti""",7
"""Q189125""","""transgender""",22


genderLabel,fraction
str,f64
"""male""",0.746305
"""female""",0.250697
"""non-binary""",0.001095
"""trans woman""",0.001093
"""trans man""",0.000248
…,…
"""apagender""",0.000002
"""neutral sex""",0.000002
"""demigirl""",0.000002


In [37]:
genderChart = (
    alt.Chart(artistGenderGrouped).mark_bar().encode(
    y="artist",
    x="genderLabel",
    )
    .properties(width=500, title="Gender Distribution of Wikidata")
)
genderChart.encoding.x.title = "Genders"
genderChart.encoding.y.title = "Counts"
genderChart.show()



genderBase = alt.Chart(artistGenderGrouped).encode(
    alt.Theta("artist:Q").stack(True),
    alt.Color("genderLabel:N").legend(None)
    )
pie = genderBase.mark_arc(outerRadius=120)
text = genderBase.mark_text(radius=140, size=20).encode(text="genderLabel:N")
pie+text

alt.Chart(...)

alt.LayerChart(...)